# S08 - RNN & LSTM
## Solutions

### Exercise 1 (Easy)
Create a simple RNN cell and pass a sequence through it.

In [ ]:
import torch
import torch.nn as nn

rnn = nn.RNN(input_size=10, hidden_size=20, batch_first=False)
x = torch.randn(5, 1, 10)  # (seq_len, batch, input_size)
h0 = torch.zeros(1, 1, 20)  # (num_layers, batch, hidden_size)

output, h_n = rnn(x, h0)
print(f"Output shape: {output.shape}")  # (5, 1, 20)
print(f"Hidden shape: {h_n.shape}")  # (1, 1, 20)

**Explanation:**

We create a vanilla RNN using `nn.RNN` with an input size of 10 and a hidden size of 20. The input tensor `x` has shape `(seq_len=5, batch=1, input_size=10)`, representing a sequence of 5 time steps for a single sample, where each time step has a 10-dimensional feature vector. We also initialize the hidden state `h0` to zeros with shape `(num_layers=1, batch=1, hidden_size=20)`.

When we call `rnn(x, h0)`, the RNN processes each time step sequentially. At each step, it applies the recurrence relation `h_t = tanh(W_ih * x_t + W_hh * h_{t-1} + bias)`, updating the hidden state based on the current input and the previous hidden state.

The RNN returns two things: `output` contains the hidden states at every time step, so its shape is `(5, 1, 20)` -- one 20-dimensional hidden vector per time step. `h_n` contains only the final hidden state, with shape `(1, 1, 20)`. In practice, `h_n` is identical to the last time step of `output`.

### Exercise 2 (Easy)
Create an LSTM and compare output shapes with vanilla RNN.

In [ ]:
lstm = nn.LSTM(input_size=10, hidden_size=20, batch_first=False)
x = torch.randn(5, 1, 10)
h0 = torch.zeros(1, 1, 20)
c0 = torch.zeros(1, 1, 20)

output, (h_n, c_n) = lstm(x, (h0, c0))
print(f"Output shape: {output.shape}")  # (5, 1, 20)
print(f"Hidden shape: {h_n.shape}")  # (1, 1, 20)
print(f"Cell shape: {c_n.shape}")  # (1, 1, 20)

**Explanation:**

The LSTM is created with the same dimensions as the RNN (input_size=10, hidden_size=20) so we can compare them directly. The key difference is that an LSTM maintains two internal states instead of one: the hidden state `h` and the cell state `c`. This is why we initialize both `h0` and `c0`, and pass them as a tuple `(h0, c0)`.

Internally, the LSTM uses four gates (forget, input, cell candidate, and output) at each time step. The forget gate decides what information to discard from the cell state, the input gate decides what new information to store, and the output gate controls what part of the cell state is exposed as the hidden state. This gating mechanism is what allows LSTMs to learn long-range dependencies, solving the vanishing gradient problem that affects vanilla RNNs.

The output shapes are the same as the vanilla RNN: `output` is `(5, 1, 20)` and `h_n` is `(1, 1, 20)`. However, the LSTM additionally returns `c_n` with shape `(1, 1, 20)`, which is the final cell state. The cell state acts as a long-term memory highway that can carry information across many time steps with minimal transformation.

### Exercise 3 (Medium)
Build an LSTM-based sentiment classifier.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        embedded = self.embedding(x)  # (batch, seq, embed)
        output, (h_n, c_n) = self.lstm(embedded)
        # Use last hidden state
        last_hidden = h_n.squeeze(0)  # (batch, hidden)
        return self.fc(last_hidden)

# Test
model = LSTMClassifier(vocab_size=1000, embed_dim=100, hidden_dim=128, num_classes=2)
x = torch.randint(0, 1000, (4, 20))  # batch=4, seq_len=20
out = model(x)
print(f"Output shape: {out.shape}")  # (4, 2)

**Explanation:**

This exercise builds a complete text classification pipeline using an LSTM. The model has three layers stacked sequentially: an embedding layer, an LSTM layer, and a fully connected (linear) layer.

In the `__init__` method, `nn.Embedding(vocab_size, embed_dim)` creates a lookup table that converts word indices into dense vectors. `nn.LSTM(embed_dim, hidden_dim, batch_first=True)` processes the sequence of embeddings, and `nn.Linear(hidden_dim, num_classes)` maps the final representation to class scores.

In the `forward` method, the input `x` (a batch of integer sequences) first passes through the embedding layer, producing a tensor of shape `(batch, seq_len, embed_dim)`. The LSTM processes this sequence and returns outputs for every time step plus the final hidden and cell states. For classification, we only need the final hidden state `h_n`, which encodes a summary of the entire sequence. We squeeze out the layer dimension with `.squeeze(0)` and pass it through the linear layer to get class logits.

The test creates a model with a vocabulary of 1000 tokens, 100-dimensional embeddings, 128 hidden units, and 2 output classes (e.g., positive/negative sentiment). A batch of 4 sequences, each of length 20, produces an output of shape `(4, 2)` -- two class scores per sample. In a real scenario, you would apply a cross-entropy loss to these logits during training.

### Exercise 4 (Medium)
Implement a character-level language model with LSTM.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        output, hidden = self.lstm(embedded, hidden)
        logits = self.fc(output)
        return logits, hidden

# Simple test
text = "hello world"
chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}

model = CharLSTM(len(chars), embed_dim=32, hidden_dim=64)
x = torch.tensor([[char2idx[c] for c in text[:-1]]])
logits, _ = model(x)
print(f"Logits shape: {logits.shape}")  # (1, 10, vocab_size)

**Explanation:**

A character-level language model predicts the next character given a sequence of previous characters. The architecture is similar to Exercise 3, but with a crucial difference: instead of producing one classification output per sequence, we produce one prediction per time step.

The model uses `nn.Embedding` to map each character index to a dense vector, `nn.LSTM` to process the sequence and capture temporal dependencies, and `nn.Linear(hidden_dim, vocab_size)` to project each hidden state back to a distribution over all characters in the vocabulary. Notice the output layer maps to `vocab_size` (not `num_classes`), because we are predicting which character comes next from the entire character vocabulary.

The `forward` method accepts an optional `hidden` parameter, which allows the model to continue generating from where it left off. This is essential during text generation: you feed one character at a time and carry the hidden state forward.

In the test section, we build a small vocabulary from the string "hello world", create character-to-index and index-to-character mappings, and feed the first 10 characters (all except the last) as input. The output logits have shape `(1, 10, vocab_size)` -- for each of the 10 input positions, the model outputs a score for every character in the vocabulary. During training, we would compare these predictions against the actual next characters using cross-entropy loss. During generation, we would sample from the predicted distribution at each step.

### Exercise 5 (Hard)
Implement a bidirectional LSTM for sequence labeling (e.g., POS tagging).

*Research: BiLSTM processes sequence in both directions and concatenates outputs.*

In [ ]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, 
                            bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, num_tags)  # *2 for bidirectional
    
    def forward(self, x):
        embedded = self.embedding(x)  # (batch, seq, embed)
        output, _ = self.lstm(embedded)  # (batch, seq, hidden*2)
        tag_scores = self.fc(output)  # (batch, seq, num_tags)
        return tag_scores

# Test
model = BiLSTMTagger(vocab_size=1000, embed_dim=100, hidden_dim=128, num_tags=10)
x = torch.randint(0, 1000, (4, 15))  # batch=4, seq_len=15
out = model(x)
print(f"Output shape: {out.shape}")  # (4, 15, 10) - tag scores per position

**Explanation:**

A bidirectional LSTM (BiLSTM) is particularly well-suited for sequence labeling tasks like POS tagging because each tag prediction can benefit from context on both sides of the word, not just the left context.

The key difference from previous exercises is `bidirectional=True` in the LSTM constructor. This makes PyTorch run two separate LSTMs internally: one processes the sequence left-to-right and the other right-to-left. At each time step, their hidden states are concatenated, which is why the output dimension becomes `hidden_dim * 2`. The linear layer must therefore accept `hidden_dim * 2` as its input size.

In the `forward` method, after embedding the input tokens, the BiLSTM returns `output` with shape `(batch, seq_len, hidden_dim * 2)`. Unlike the classifier in Exercise 3 where we only used the final hidden state, here we need predictions at every position in the sequence (one tag per word), so we pass the full `output` tensor through the linear layer. This produces `tag_scores` with shape `(batch, seq_len, num_tags)`.

The test creates a model with 10 possible tags and sequences of length 15. The output `(4, 15, 10)` means that for each of the 4 sequences and each of the 15 positions, we get a score for all 10 possible tags. During training, we would apply cross-entropy loss at each position, and during inference we would take the argmax at each position to get the predicted tag sequence.